# Module 4, Demo lab 2: Tests & confidence intervals

Not graded · Follow along · pairs with [Demo lab 1 (simulate & compare)](module-4-lab-demo) and [Project 4](../projects/project-4).

Demo 1 described groups. This demo runs the inference that addresses the *by-chance* explanation and estimates effects with 95% CIs. Each cell re-simulates its own small dataset so it stands alone. It has two parts: two-group tests (Part A) and more than two groups (Part B). Confounding is handled with descriptive statistics and subclassification in [Demo 1](module-4-lab-demo), not with modeling, so it does not appear here. Match each method to its row on the [methods map](../methods).

| Method | When | R |
|---|---|---|
| Welch t-test | 2 groups, numerical | `t.test(y ~ group, var.equal = FALSE)` |
| Two-proportion / chi-square | 2 groups, binary | `prop.test()` / `chisq.test()` |
| Paired t-test | before/after, numerical | `t.test(after, before, paired = TRUE)` |
| McNemar | before/after, binary | `mcnemar.test()` |
| One-way ANOVA | 3+ groups, numerical | `anova(lm(y ~ group))` |
| Chi-square (independence) | groups x categorical | `chisq.test(table(group, outcome))` |


## Code anatomy legend (demo notebooks only)

As you read through the demo and see R code, try to break it down into the following 4 components:

| Color | Meaning in code |
|-------|-----------------|
| <span style="color:#2563eb">Blue</span> | R functions and syntax (`t.test`, `<-`, `()`, `~`) |
| <span style="color:#059669">Green</span> | Names you created (variables, data frames) |
| <span style="color:#dc2626">Red</span> | Values to change for your question or data |
| <span style="color:#7c3aed">Purple</span> | Important output to read carefully |


*Note:* Colab may highlight R syntax in its own colors. My anatomy colors appear only on my website, not in Colab.

In [ ]:
library(tidyverse)

# Part A, Two-group tests

## Welch two-sample t-test (numerical, two groups)

Use this when the response variable is numerical and you compare two independent groups. $H_0: \mu_1 = \mu_2$ (the two population means are equal).

Code anatomy of `t.test(sleep ~ group, data = df, var.equal = FALSE)`:

- `t.test(...)` is the function that runs the two-sample t-test.
- `sleep ~ group` is a formula: the numerical response on the left of `~`, the two-level grouping variable on the right. Read it as "sleep by group."
- `data = df` says which data frame holds those columns.
- `var.equal = FALSE` requests the Welch version, which does not assume the two groups have equal variances. This is the safe default for this course.

Conditions: independent observations, and each group roughly normal or large enough (about 20 or more per group).

In [ ]:
set.seed(42)
df <- tibble(
  group = factor(rep(c("Control", "Treatment"), each = 30),
                 levels = c("Control", "Treatment")),
  sleep = c(rnorm(30, 6.8, 1.0), rnorm(30, 7.5, 1.0))
)
t.test(sleep ~ group, data = df, var.equal = FALSE)

Reading the output: compare `p-value` to $\alpha = 0.05$ for the by-chance decision, and read the `95 percent confidence interval` as a plausible range for the difference in mean sleep (Treatment minus Control). The true difference built into this simulation is 7.5 − 6.8 = 0.7, and the interval typically covers it.

## Two proportions (binary, two groups)

Use this when the response variable is binary (yes/no) and you compare two independent groups. $H_0: p_1 = p_2$ (the two population proportions are equal).

Code anatomy of `prop.test(c(sum(x1), sum(x2)), c(80, 80), correct = FALSE)`:

- `prop.test(...)` tests whether two proportions are equal and returns a CI for $p_1 - p_2$.
- the first argument is the count of "yes" in each group, `c(sum(x1), sum(x2))`.
- the second argument is the two group sizes, `c(80, 80)`.
- `correct = FALSE` turns off the continuity correction so the result lines up with `chisq.test()`.

Code anatomy of `chisq.test(tab)`:

- `tab` is a 2x2 table of counts (group by outcome), built with `table(...)`.
- `chisq.test(tab)` runs the chi-square test of independence on that table; $H_0$ is that group and outcome are independent (no association).
- `chisq.test(tab)$expected` prints the expected counts; the condition is that all expected counts are at least 5.

For two groups, `prop.test()` and `chisq.test()` give the same p-value; use `prop.test()` when you also want the CI for the difference in proportions.

In [ ]:
set.seed(42)
x1 <- rbinom(80, 1, 0.45)   # placebo recoveries
x2 <- rbinom(80, 1, 0.65)   # drug recoveries

prop.test(c(sum(x1), sum(x2)), c(80, 80), correct = FALSE)

tab <- table(group = rep(c("Placebo", "Drug"), each = 80),
             recovered = c(x1, x2))
chisq.test(tab)$expected   # all should be >= 5
chisq.test(tab)

## Paired t-test (numerical, before/after)

Same people measured twice, so test the mean of the within-person differences. $H_0$: mean change = 0.

Code anatomy of `t.test(after, before, paired = TRUE)`: `after` and `before` are two numeric vectors measured on the same units, and `paired = TRUE` tells R to test the mean of the differences (after minus before) rather than compare two independent groups.

In [ ]:
set.seed(202)
before <- rnorm(25, 150, 12)
after  <- before - rnorm(25, 6, 4)
t.test(after, before, paired = TRUE)

## McNemar's test (binary, before/after)

Paired yes/no answers (same people answer twice). Build the 2x2 table of paired responses; `mcnemar.test()` tests the discordant (off-diagonal) cells. An ordinary two-proportion test would be wrong because the answers are not independent.

In [ ]:
# 100 voters: support (Yes/No) before and after a debate
paired_tab <- matrix(c(30, 12, 5, 53), nrow = 2,
                     dimnames = list(before = c("Yes", "No"),
                                     after  = c("Yes", "No")))
paired_tab
mcnemar.test(paired_tab)

# Part B, More than two groups

When there are three or more groups, use one method that tests all groups at once instead of running many two-group tests.

## One-way ANOVA (numerical, 3+ groups)

$H_0: \mu_A = \mu_B = \mu_C$ (all group means equal).

Code anatomy of `anova(lm(bp_drop ~ clinic, data = three))`:

- `lm(bp_drop ~ clinic, data = three)` fits a linear model: the numerical response `bp_drop` explained by the grouping factor `clinic`. Read the formula as "bp_drop by clinic."
- `anova(fit)` turns that fitted model into the ANOVA table.
- read `Pr(>F)`: a small value says at least one group mean differs, not which one.

Conditions: independent observations, roughly normal within groups, and similar spread across groups.

In [ ]:
set.seed(101)
three <- tibble(
  clinic  = factor(rep(c("A", "B", "C"), each = 40)),
  bp_drop = c(rnorm(40, 8, 3), rnorm(40, 10, 3), rnorm(40, 13, 3))
)
fit <- lm(bp_drop ~ clinic, data = three)
anova(fit)

## Chi-square test (categorical, including 3+ groups)

For a categorical response across two or more groups, use the chi-square test on a table of counts.

`chisq.test()` has two uses, and it is worth keeping them straight:

- Goodness of fit: one categorical variable compared to a set of expected proportions. Example: are the four blood types in the proportions 0.45, 0.40, 0.11, 0.04? Call `chisq.test(counts, p = expected_props)`.
- Test of independence: two categorical variables in a contingency table, testing whether they are associated. Example: does recovery (yes/no) depend on clinic (A/B/C)? Call `chisq.test(table(group, outcome))`. This is the 2x2 case from Part A, now extended to a larger group-by-outcome table.

The cell below uses the test of independence for three clinics. $H_0$: clinic and recovery are independent (no association). Condition: all expected counts at least 5 (check with `$expected`).

In [ ]:
set.seed(404)
clinic_df <- tibble(
  clinic    = factor(rep(c("A", "B", "C"), each = 60)),
  recovered = factor(c(rbinom(60, 1, 0.55), rbinom(60, 1, 0.60), rbinom(60, 1, 0.75)),
                     labels = c("No", "Yes"))
)

tab <- table(clinic_df$clinic, clinic_df$recovered)
tab
chisq.test(tab)$expected   # all should be >= 5
chisq.test(tab)

## Validate a test against known truth (simulation)

Generate many datasets where the two groups have the same mean (null true) and record each p-value. The rejection rate should be near $\alpha = 0.05$ and the p-value histogram roughly uniform, a check that your simulation matches the stated null.

In [ ]:
set.seed(123)
n_sim <- 1000
pvals <- numeric(n_sim)
for (i in seq_len(n_sim)) {
  g1 <- rnorm(30, 5, 2)
  g2 <- rnorm(30, 5, 2)   # same mean: null is TRUE
  pvals[i] <- t.test(g1, g2)$p.value
}
mean(pvals < 0.05)   # should be near 0.05

ggplot(tibble(p = pvals), aes(x = p)) +
  geom_histogram(binwidth = 0.05, boundary = 0) +
  labs(title = "P-values under H0 (1000 sims) -- roughly uniform",
       x = "p-value", y = "Count")

## Wrap-up

Part A covered the two-group tests (t.test, prop.test/chisq.test, and their paired versions) and Part B covered more than two groups (one-way ANOVA after `lm`, and the chi-square test). Confounding is not a test: in this course you handle it with descriptive statistics and subclassification (Demo 1). For [Project 4](../projects/project-4) you only need one two-group comparison: simulate it (Demo 1), then run the matching test from Part A and report the p-value and 95% CI with a one-sentence causal-or-association conclusion.